# <center> VAI Store - Engenharia de Variáveis </center>

---

Com base nos **_insights_** da Análise Exploratória de Dados, nosso objetivo nesta etapa é **traduzir os padrões que descobrimos** (como sazonalidade e diferença entre filiais) em _**features**_ que o modelo possa usar para prever a demanda dos produtos.

### **1. Configuração e Imports**

Visão Geral: Vamos começar importando as bibliotecas essenciais (pandas para manipulação de dados e numpy para operações numéricas). Também definiremos os caminhos relativos para os arquivos de dados brutos, facilitando a organização.

In [196]:
import pandas as pd
import numpy as np
import os

# Define os caminhos relativos para "subir um nível" (../)
# e depois entrar na pasta 'data'
RAW_DATA_PATH = os.path.join('..', 'data', 'raw')
VENDAS_FILE = os.path.join(RAW_DATA_PATH, 'vendas.csv')
PRODUTO_FILE = os.path.join(RAW_DATA_PATH, 'produto.csv')

# Define o caminho de saída da mesma forma
PROCESSED_DATA_PATH = os.path.join('..', 'data', 'processed')

# O arquivo de saída agora é .parquet
OUTPUT_FILE = os.path.join(PROCESSED_DATA_PATH, 'feature_engineered_dataset.parquet')

# Garante que o diretório de saída exista no local correto
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Imprime os caminhos absolutos para verificação
print(f"Lendo dados de: {os.path.abspath(RAW_DATA_PATH)}")
print(f"Salvando dados em: {os.path.abspath(PROCESSED_DATA_PATH)}")

Lendo dados de: c:\Users\felip\OneDrive\Documentos\GitHub\Grupo6-ProjetoFinal\data\raw
Salvando dados em: c:\Users\felip\OneDrive\Documentos\GitHub\Grupo6-ProjetoFinal\data\processed


### **2. Funções Auxiliares de Carga e Limpeza**

Visão Geral: Para manter o notebook limpo, definiremos duas funções auxiliares. A load_csv lida com os diferentes tipos de codificação (utf-8 ou latin1) que podemos encontrar. A clean_sku padroniza a coluna SKU, que observamos ter espaços extras e tipo de dado incorreto, garantindo que as chaves de merge sejam consistentes.

In [197]:
def clean_sku(sku_series):
    """Limpa a coluna SKU, removendo espaços e convertendo para inteiro."""
    return pd.to_numeric(sku_series.astype(str).str.strip(), errors='coerce').astype('Int64')

def load_csv(file_path):
    """Tenta carregar um CSV com encoding 'utf-8', e usa 'latin1' como fallback."""
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='latin1')

### **3. Carregar e Agregar Dados de Vendas**

Visão Geral: O vendas.csv bruto está em nível transacional (várias vendas por dia). Para prever a demanda diária, precisamos agregá-lo. Vamos agrupar os dados por DATA_ATEND, SKU e FILIAL (que transformaremos em SHOPPING = 1 ou 0). Calcularemos a QTD_TOTAL, FATUR_TOTAL e o número de CLIENTES_UNICOS para cada um desses grupos. Este será nosso DataFrame base.

In [198]:
print(f"Carregando e agregando {VENDAS_FILE}...")

df_vendas = load_csv(VENDAS_FILE)

# Converte data e limpa SKU
df_vendas['DATA_ATEND'] = pd.to_datetime(df_vendas['DATA_ATEND'])
df_vendas['SKU'] = clean_sku(df_vendas['SKU'])

# Cria a feature 'SHOPPING' (One-Hot Encoding da FILIAL)
df_vendas['SHOPPING'] = (df_vendas['FILIAL'] == 'SHOPPING').astype(int)

# Agrega os dados
df_agg = df_vendas.groupby(['DATA_ATEND', 'SKU', 'SHOPPING']).agg(
    QTD_TOTAL=('QTD_VENDA', 'sum'),
    FATUR_TOTAL=('FATUR_VENDA', 'sum'),
    CLIENTES_UNICOS=('CLI_CPF', 'nunique')
).reset_index()

# Remove SKUs nulos que podem ter sido criados pela limpeza
df_agg = df_agg.dropna(subset=['SKU'])

print(f"Dados de vendas agregados. Shape: {df_agg.shape}")

Carregando e agregando ..\data\raw\vendas.csv...
Dados de vendas agregados. Shape: (30332, 6)


### **4. Adicionar Contexto do Produto (Categorias)**

Visão Geral: Para o modelo aprender padrões entre produtos (como sugerido na dica "Verificar tipo de produto"), precisamos mesclar as CATEGORIAs e SUBCATEGORIAs do produto.csv. Vamos carregar o arquivo de produtos, limpá-lo, tratar valores ausentes como 'DESCONHECIDA' e juntá-lo ao nosso DataFrame agregado df_agg.

In [199]:
df_prod = load_csv(PRODUTO_FILE)
df_prod['SKU'] = clean_sku(df_prod['SKU'])

# 1. Remove SKUs que falharam na limpeza
df_prod = df_prod.dropna(subset=['SKU'])

# 2. Seleciona colunas de contexto
df_prod_context = df_prod[['SKU', 'CATEGORIA', 'SUBCATEGORIA']].copy()

# 3. Aplica fillna apenas nas colunas de string
df_prod_context['CATEGORIA'] = df_prod_context['CATEGORIA'].fillna('DESCONHECIDA')
df_prod_context['SUBCATEGORIA'] = df_prod_context['SUBCATEGORIA'].fillna('DESCONHECIDA')

# Remove duplicatas de SKU para garantir um merge limpo
df_prod_context = df_prod_context.drop_duplicates(subset=['SKU'])

# Mescla o contexto ao dataset agregado
df_merged = pd.merge(df_agg, df_prod_context, on='SKU', how='left')

# Garante que mesmo SKUs sem correspondência em produto.csv tenham categoria
# (Ex: SKUs de vendas que não existem no cadastro de produtos)
df_merged[['CATEGORIA', 'SUBCATEGORIA']] = df_merged[['CATEGORIA', 'SUBCATEGORIA']].fillna('DESCONHECIDA')

print(f"Contexto do produto mesclado. Shape: {df_merged.shape}")

Contexto do produto mesclado. Shape: (30332, 8)


### **5. Criar "Scaffold" (Grid de Datas Completo)**

Visão Geral: Nossos dados atuais são "esparsos" (só têm linhas em dias que houve venda). O modelo precisa ver os dias de "venda zero" para entender a sazonalidade corretamente. Vamos criar um "gabarito" (scaffold) com uma linha para cada dia (do min ao max) e para cada combinação SKU/SHOPPING que existe.

In [200]:
group_cols = ['SKU', 'SHOPPING']

# Encontra o range de datas e os grupos únicos
min_date = df_merged['DATA_ATEND'].min()
max_date = df_merged['DATA_ATEND'].max()
date_range = pd.date_range(min_date, max_date, freq='D')
unique_groups = df_merged[group_cols].drop_duplicates()

# Cria o scaffold
df_scaffold = pd.MultiIndex.from_product(
    [unique_groups[col] for col in group_cols] + [date_range],
    names=group_cols + ['DATA_ATEND']
).to_frame(index=False)

# Junta os dados reais ao scaffold
df_full = pd.merge(df_scaffold, df_merged, on=group_cols + ['DATA_ATEND'], how='left')

print(f"Scaffold criado. Shape: {df_full.shape}")

Scaffold criado. Shape: (4911440, 8)


### **6. Preencher Dados Ausentes (Pós-Scaffold)**

Visão Geral: O scaffold criou NaNs em dias sem vendas. Agora vamos preenchê-los corretamente.

1. Métricas (QTD_TOTAL, FATUR_TOTAL, CLIENTES_UNICOS): Serão preenchidas com 0.

2. Contexto (CATEGORIA, SUBCATEGORIA): Serão preenchidas com o último valor válido (ffill) e depois com o próximo (bfill), garantindo que todos os SKUs tenham suas categorias preenchidas em todos os dias.

In [201]:
df_full = df_full.sort_values(by=group_cols + ['DATA_ATEND'])

# Preenche métricas com 0
metric_cols = ['QTD_TOTAL', 'FATUR_TOTAL', 'CLIENTES_UNICOS']
df_full[metric_cols] = df_full[metric_cols].fillna(0)

# Preenche contexto (ffill = 'forward fill', bfill = 'backward fill')
context_cols = ['CATEGORIA', 'SUBCATEGORIA']
df_full[context_cols] = df_full.groupby(group_cols)[context_cols].ffill().bfill()

# Remove SKUs que possam ter ficado sem categoria (raro)
df_full = df_full.dropna(subset=context_cols)

print("Preenchimento concluído.")

Preenchimento concluído.


### **7. Engenharia de Features de Calendário**

Visão Geral: Agora que temos um DataFrame denso, podemos criar features de calendário para capturar sazonalidades. Vamos extrair DIA_SEMANA, MES, ANO, DIA_DO_MES e criar features de "dia de pagamento" (INICIO_MES, FIM_MES), que são fortes indicadores de venda no varejo.

In [202]:
date_col = df_full['DATA_ATEND']

df_full['DIA_SEMANA'] = date_col.dt.dayofweek
df_full['DIA_DO_MES'] = date_col.dt.day
df_full['DIA_DO_ANO'] = date_col.dt.dayofyear
df_full['MES'] = date_col.dt.month
df_full['SEMANA_DO_ANO'] = date_col.dt.isocalendar().week.astype(int)
df_full['ANO'] = date_col.dt.year

df_full['INICIO_MES'] = (date_col.dt.day <= 5).astype(int)
df_full['FIM_MES'] = (date_col.dt.day >= 28).astype(int)

print("Features de calendário criadas.")

Features de calendário criadas.


### **8. Engenharia de Features de Feriados**

Visão Geral: A simples sazonalidade de calendário (Etapa 7) não captura eventos-chave como a Páscoa (que é móvel) ou o período de compras pré-Natal. Aqui, vamos criar features binárias para EH_FERIADO e, o mais importante, para a "antecipação" (os 14 dias antes) da Páscoa e do Natal, que são os períodos que impulsionam as vendas.

In [203]:
import holidays

# Pega os anos únicos do nosso dataset (que já está no df_full)
anos = df_full['ANO'].unique()

# Cria um objeto de feriados do Brasil para os anos relevantes
br_holidays = holidays.Brazil(years=anos)

# Converte o dicionário de feriados em um DataFrame
df_holidays = pd.DataFrame(br_holidays.items(), columns=['DATA_ATEND', 'NOME_FERIADO'])
df_holidays['DATA_ATEND'] = pd.to_datetime(df_holidays['DATA_ATEND'])
df_holidays['EH_FERIADO'] = 1

# 1. Mapeia datas da Páscoa
pascoa_datas = {}
for date, name in br_holidays.items():
    if name == 'Páscoa':
        pascoa_datas[date.year] = pd.to_datetime(date)

# 2. Mapeia data do Natal
natal_datas = {ano: pd.to_datetime(f'{ano}-12-25') for ano in anos}

# 3. Mapeia data do Ano Novo (Jan 1 do ano SEGUINTE)
anos_com_seguinte = np.append(anos, anos.max() + 1)
ano_novo_datas = {ano: pd.to_datetime(f'{ano+1}-01-01') for ano in anos_com_seguinte}

df_full['DATA_PASCOA'] = df_full['ANO'].map(pascoa_datas)
df_full['DATA_NATAL'] = df_full['ANO'].map(natal_datas)
df_full['DATA_ANO_NOVO'] = df_full['ANO'].map(ano_novo_datas)

# Força a conversão de tipo ANTES da subtração
data_pascoa_ts = pd.to_datetime(df_full['DATA_PASCOA'])
data_natal_ts = pd.to_datetime(df_full['DATA_NATAL'])
data_ano_novo_ts = pd.to_datetime(df_full['DATA_ANO_NOVO'])
data_atend_ts = pd.to_datetime(df_full['DATA_ATEND'])

# Calcula os dias restantes até esses eventos
df_full['DIAS_ATE_PASCOA'] = (data_pascoa_ts - data_atend_ts).dt.days
df_full['DIAS_ATE_NATAL'] = (data_natal_ts - data_atend_ts).dt.days
df_full['DIAS_ATE_ANO_NOVO'] = (data_ano_novo_ts - data_atend_ts).dt.days

# Cria as features de antecipação (Flags de Evento)
df_full['ANTECIPACAO_PASCOA_14D'] = ((df_full['DIAS_ATE_PASCOA'] >= 0) & (df_full['DIAS_ATE_PASCOA'] <= 14)).astype(int)
df_full['ANTECIPACAO_NATAL_21D'] = ((df_full['DIAS_ATE_NATAL'] >= 0) & (df_full['DIAS_ATE_NATAL'] <= 21)).astype(int)
df_full['ANTECIPACAO_ANO_NOVO_7D'] = ((df_full['DIAS_ATE_ANO_NOVO'] >= 0) & (df_full['DIAS_ATE_ANO_NOVO'] <= 7)).astype(int)

# Esta é a chave: a flag só é 1 se AMBAS as condições forem verdadeiras
# (O nome da categoria "Bacalhau & Pescados" deve ser exato)
df_full['INTERACAO_PASCOA_BACALHAU'] = (
    (df_full['ANTECIPACAO_PASCOA_14D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)

df_full['INTERACAO_NATAL_BACALHAU'] = (
    (df_full['ANTECIPACAO_NATAL_21D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)

df_full['INTERACAO_ANO_NOVO_BACALHAU'] = (
    (df_full['ANTECIPACAO_ANO_NOVO_7D'] == 1) &
    (df_full['CATEGORIA'] == 'Bacalhau & Pescados')
).astype(int)

# Junta a feature EH_FERIADO
df_full = pd.merge(df_full, df_holidays[['DATA_ATEND', 'EH_FERIADO']], on='DATA_ATEND', how='left')
df_full['EH_FERIADO'] = df_full['EH_FERIADO'].fillna(0).astype(int)

# Limpa colunas auxiliares
df_full = df_full.drop(columns=[
    'DATA_PASCOA', 'DATA_NATAL', 'DATA_ANO_NOVO',
    'DIAS_ATE_PASCOA', 'DIAS_ATE_NATAL', 'DIAS_ATE_ANO_NOVO'
])

# Trata casos onde a Páscoa/Ano Novo podem não estar no map (raro)
df_full['ANTECIPACAO_PASCOA_14D'] = df_full['ANTECIPACAO_PASCOA_14D'].fillna(0).astype(int)
df_full['ANTECIPACAO_ANO_NOVO_7D'] = df_full['ANTECIPACAO_ANO_NOVO_7D'].fillna(0).astype(int)
df_full = df_full.fillna(0) # Limpa NaNs das interações

print("Features de feriados (refinadas com interações) criadas.")

Features de feriados (refinadas com interações) criadas.


### **9. Engenharia de Features de Preço**

Visão Geral: Além de criar o PRECO_MEDIO_DIA, agora também criaremos features de preço relativo. O modelo precisa saber se um preço é "caro" ou "barato" para aquele produto. Para isso, calculamos o PRECO_MEDIO_HIST de cada SKU e, a partir dele, criamos PRECO_RELATIVO (se está acima ou abaixo da média) e EM_PROMOCAO (uma flag binária).

In [204]:
# 1. Preço do dia (evita divisão por zero)
df_full['PRECO_MEDIO_DIA'] = df_full['FATUR_TOTAL'] / (df_full['QTD_TOTAL'] + 1e-6)

# 2. Preenche o preço nos dias de não-venda (0) com o último preço válido
df_full['PRECO_MEDIO_DIA'] = df_full.groupby(group_cols)['PRECO_MEDIO_DIA'].transform(
    lambda x: x.replace(0, np.nan).ffill().bfill()
)
df_full['PRECO_MEDIO_DIA'] = df_full['PRECO_MEDIO_DIA'].fillna(0) # Se um SKU nunca foi vendido

# 3. Calcula o preço médio histórico (ignorando dias de não-venda)
preco_medio_hist = df_full.groupby(group_cols)['PRECO_MEDIO_DIA'].transform(
    lambda x: x[x > 0].mean()
)

# 4. Preço relativo (Ex: 1.0 = normal, 0.8 = 20% desconto)
df_full['PRECO_RELATIVO'] = df_full['PRECO_MEDIO_DIA'] / (preco_medio_hist + 1e-6)

# 5. Flag de Promoção (Ex: 10% abaixo da média histórica)
df_full['EM_PROMOCAO'] = (df_full['PRECO_RELATIVO'] <= 0.9).astype(int)

# Limpa NaNs caso um produto nunca tenha tido um preço > 0
df_full['PRECO_RELATIVO'] = df_full['PRECO_RELATIVO'].fillna(0)
df_full['EM_PROMOCAO'] = df_full['EM_PROMOCAO'].fillna(0)

print("Features de preço criadas.")

Features de preço criadas.


### **10. Engenharia de Features de Lag (Defasagem)**

Visão Geral: Aqui damos "memória" ao modelo. Vamos criar features de lag (valor de X dias atrás) para nossas métricas principais. Isso é fundamental para capturar a forte sazonalidade semanal (lags 7, 14, 21, 28). Isso só funciona corretamente porque criamos o scaffold denso.

In [205]:
targets = ['QTD_TOTAL', 'FATUR_TOTAL', 'CLIENTES_UNICOS']
lags = [7, 14, 21, 28]

for target in targets:
    for lag in lags:
        col_name = f'{target}_LAG_{lag}'
        df_full[col_name] = df_full.groupby(group_cols)[target].shift(lag)

print("Features de lag criadas.")

Features de lag criadas.


### **11. Engenharia de Features de Rolling Window (Janela Móvel)**

Visão Geral: Além de dias específicos (lags), o trend recente é importante. Vamos calcular a média, mediana e desvio padrão dos últimos 7, 14 e 28 dias. Importante: Usamos .shift(1) antes do .rolling() para evitar data leakage (usar os dados de hoje para prever hoje). Isso garante que o modelo só use informações do passado.

In [206]:
windows = [7, 14, 28]

for target in targets:
    # shift(1) é crucial para evitar data leakage
    shifted_data = df_full.groupby(group_cols)[target].shift(1)
    
    for window in windows:
        df_full[f'{target}_ROLLING_MEAN_{window}'] = shifted_data.rolling(window, min_periods=1).mean()
        df_full[f'{target}_ROLLING_MEDIAN_{window}'] = shifted_data.rolling(window, min_periods=1).median()
        df_full[f'{target}_ROLLING_STD_{window}'] = shifted_data.rolling(window, min_periods=1).std()

print("Features de rolling window criadas.")

Features de rolling window criadas.


### **12. Limpeza Final e Salvamento**

Visão Geral: As features de lag e rolling criaram NaNs no início da série temporal de cada produto (o que é normal). Vamos preenchê-los com 0 (assumindo que não há histórico anterior) e salvar o dataset final e pronto para o modelo no caminho data/processed que definimos no início.

In [207]:
print("Iniciando limpeza final e salvamento...")

df_final = df_full.fillna(0)

# Opcional: Converte tipos de dados para otimizar uso de memória
for col in df_final.columns:
    if col.startswith(('QTD_', 'FATUR_', 'CLIENTES_')):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='float')
    
    if col.startswith(('DIA_', 'MES', 'ANO', 'SEMANA_', 'INICIO_', 'FIM_', 'SHOPPING', 'EH_FERIADO', 'ANTECIPACAO_', 'EM_PROMOCAO', 'INTERACAO_')):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='integer')
        
    if col.startswith(('PRECO_RELATIVO')):
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce', downcast='float')

# Salva o arquivo em Parquet
print("Salvando arquivo em formato Parquet...")
df_final.to_parquet(OUTPUT_FILE, index=False, engine='pyarrow')

print(f"\nEngenharia de features concluída! Dataset salvo em:\n{os.path.abspath(OUTPUT_FILE)}")
print("\n--- Informações do DataFrame Final ---")
df_final.info()

Iniciando limpeza final e salvamento...
Salvando arquivo em formato Parquet...


KeyboardInterrupt: 